# End-to-End Sales Forecasting & Demand Intelligence System

## Project Objective
This project builds an intelligent retail sales forecasting and demand analysis system.

The system includes:
- Exploratory Data Analysis
- Time Series Feature Engineering
- Weekly and Monthly Sales Aggregation
- Time Series Decomposition
- Stationarity Testing
- SARIMA Forecasting
- Prophet Forecasting
- XGBoost Time Series Forecasting
- Model Comparison
- Category and Region-Level Forecasting
- Anomaly Detection
- Product Demand Segmentation
- Business Recommendations

## Datasets
1. Sales_data.csv — Primary Superstore sales dataset
2. vgsales.csv — Supplementary dataset for multi-source analysis

In [2]:
%pip install pandas numpy matplotlib seaborn plotly statsmodels scikit-learn xgboost prophet joblib -q

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\LENOVO\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python312\\site-packages\\prophet\\stan_model\\cmdstan-2.37.0\\stan\\lib\\stan_math\\lib\\tbb_2020.3\\include\\tbb\\internal\\_deprecated_header_message_guard.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\LENOVO\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
%python --version

UsageError: Line magic function `%python` not found (But cell magic `%%python` exists, did you mean that instead?).


In [5]:
%pip install pandas numpy matplotlib seaborn plotly statsmodels scikit-learn xgboost prophet joblib -q

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'C:\\Users\\LENOVO\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python312\\site-packages\\prophet\\stan_model\\cmdstan-2.37.0\\stan\\lib\\stan_math\\lib\\tbb_2020.3\\include\\tbb\\internal\\_deprecated_header_message_guard.h'
HINT: This error might have occurred since this system does not have Windows Long Path support enabled. You can find information on how to enable this at https://pip.pypa.io/warnings/enable-long-paths


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\LENOVO\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from xgboost import XGBRegressor
from prophet import Prophet

import joblib

pd.set_option("display.max_columns", None)

sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

Libraries imported successfully.


# Task 1 — Data Loading, Merging & Deep Exploration

The primary objective of this section is to:
- Load both datasets
- Inspect their structure
- Clean date and missing-value issues
- Engineer time-based features
- Aggregate sales at weekly and monthly levels
- Answer key business questions

In [8]:
sales = pd.read_csv("Sales_data.csv")
vgsales = pd.read_csv("vgsales.csv")

print("Primary Sales Dataset Shape:", sales.shape)
print("Video Game Sales Dataset Shape:", vgsales.shape)

Primary Sales Dataset Shape: (9800, 18)
Video Game Sales Dataset Shape: (16598, 11)


In [9]:
display(sales.head())
display(vgsales.head())

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [10]:
print("=" * 60)
print("PRIMARY SALES DATASET")
print("=" * 60)

sales.info()

print("\n" + "=" * 60)
print("VIDEO GAME SALES DATASET")
print("=" * 60)

vgsales.info()

PRIMARY SALES DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   object 
 2   Order Date     9800 non-null   object 
 3   Ship Date      9800 non-null   object 
 4   Ship Mode      9800 non-null   object 
 5   Customer ID    9800 non-null   object 
 6   Customer Name  9800 non-null   object 
 7   Segment        9800 non-null   object 
 8   Country        9800 non-null   object 
 9   City           9800 non-null   object 
 10  State          9800 non-null   object 
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   object 
 13  Product ID     9800 non-null   object 
 14  Category       9800 non-null   object 
 15  Sub-Category   9800 non-null   object 
 16  Product Name   9800 non-null   object 
 17  Sales          9800 non-null  

In [11]:
display(sales.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Row ID,9800.0,NaN,NaN,NaN,4900.5,2829.160653,1.0,2450.75,4900.5,7350.25,9800.0
Order ID,9800,4922,CA-2018-100111,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order Date,9800,1230,05/09/2017,38,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ship Date,9800,1326,26/09/2018,34,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ship Mode,9800,4,Standard Class,5859,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer ID,9800,793,WB-21850,35,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Customer Name,9800,793,William Brown,35,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Segment,9800,3,Consumer,5101,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,9800,1,United States,9800,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,9800,529,New York City,891,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
sales["Order Date"] = pd.to_datetime(
    sales["Order Date"],
    dayfirst=True,
    errors="coerce"
)

sales["Ship Date"] = pd.to_datetime(
    sales["Ship Date"],
    dayfirst=True,
    errors="coerce"
)

print(sales[["Order Date", "Ship Date"]].head())
print("\nDate Range:")
print("Start:", sales["Order Date"].min())
print("End:", sales["Order Date"].max())

  Order Date  Ship Date
0 2017-11-08 2017-11-11
1 2017-11-08 2017-11-11
2 2017-06-12 2017-06-16
3 2016-10-11 2016-10-18
4 2016-10-11 2016-10-18

Date Range:
Start: 2015-01-03 00:00:00
End: 2018-12-30 00:00:00


In [13]:
missing_values = pd.DataFrame({
    "Missing Count": sales.isnull().sum(),
    "Missing Percentage": (
        sales.isnull().mean() * 100
    ).round(2)
})

display(
    missing_values[
        missing_values["Missing Count"] > 0
    ]
)

,Missing Count,Missing Percentage
Postal Code,11,0.11


In [15]:
print("Exact duplicate rows:", sales.duplicated().sum())

print(
    "Duplicate Row IDs:",
    sales["Row ID"].duplicated().sum()
)

print(
    "Duplicate Order IDs:",
    sales["Order ID"].duplicated().sum()
)

Exact duplicate rows: 0
Duplicate Row IDs: 0
Duplicate Order IDs: 4878


Repeated Order ID values are not necessarily errors because one order may contain multiple products.

In [16]:
dtype_table = pd.DataFrame({
    "Column": sales.columns,
    "Data Type": sales.dtypes.astype(str).values,
    "Missing Values": sales.isnull().sum().values,
    "Unique Values": sales.nunique().values
})

display(dtype_table)

,Column,Data Type,Missing Values,Unique Values
0,Row ID,int64,0,9800
1,Order ID,object,0,4922
2,Order Date,datetime64[ns],0,1230
3,Ship Date,datetime64[ns],0,1326
4,Ship Mode,object,0,4
5,Customer ID,object,0,793
6,Customer Name,object,0,793
7,Segment,object,0,3
8,Country,object,0,1
9,City,object,0,529


In [17]:
sales = sales.drop_duplicates().copy()

sales = sales.dropna(
    subset=["Order Date", "Ship Date", "Sales"]
)

sales["Sales"] = pd.to_numeric(
    sales["Sales"],
    errors="coerce"
)

sales = sales.dropna(subset=["Sales"])

print("Cleaned Shape:", sales.shape)

Cleaned Shape: (9800, 18)


In [18]:
sales["Year"] = sales["Order Date"].dt.year
sales["Month"] = sales["Order Date"].dt.month
sales["Month Name"] = sales["Order Date"].dt.month_name()

sales["Week Number"] = (
    sales["Order Date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

sales["Day of Week"] = sales["Order Date"].dt.day_name()
sales["Quarter"] = sales["Order Date"].dt.quarter

display(
    sales[
        [
            "Order Date",
            "Year",
            "Month",
            "Month Name",
            "Week Number",
            "Day of Week",
            "Quarter"
        ]
    ].head()
)

,Order Date,Year,Month,Month Name,Week Number,Day of Week,Quarter
0,2017-11-08,2017,11,November,45,Wednesday,4
1,2017-11-08,2017,11,November,45,Wednesday,4
2,2017-06-12,2017,6,June,24,Monday,2
3,2016-10-11,2016,10,October,41,Tuesday,4
4,2016-10-11,2016,10,October,41,Tuesday,4


In [19]:
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


sales["Season"] = sales["Month"].apply(get_season)

display(
    sales[
        ["Order Date", "Month", "Season"]
    ].head()
)

,Order Date,Month,Season
0,2017-11-08,11,Autumn
1,2017-11-08,11,Autumn
2,2017-06-12,6,Summer
3,2016-10-11,10,Autumn
4,2016-10-11,10,Autumn


In [20]:
sales["Shipping Days"] = (
    sales["Ship Date"] - sales["Order Date"]
).dt.days

display(
    sales[
        ["Order Date", "Ship Date", "Shipping Days"]
    ].head()
)

,Order Date,Ship Date,Shipping Days
0,2017-11-08,2017-11-11,3
1,2017-11-08,2017-11-11,3
2,2017-06-12,2017-06-16,4
3,2016-10-11,2016-10-18,7
4,2016-10-11,2016-10-18,7


Multi-source analysis with vgsales.csv

In [21]:
vgsales["Year"] = pd.to_numeric(
    vgsales["Year"],
    errors="coerce"
)

vgsales["Global_Sales"] = pd.to_numeric(
    vgsales["Global_Sales"],
    errors="coerce"
)

vgsales_clean = vgsales.dropna(
    subset=["Year", "Global_Sales"]
).copy()

vgsales_clean["Year"] = (
    vgsales_clean["Year"].astype(int)
)

print("Clean Video Game Dataset Shape:", vgsales_clean.shape)

display(vgsales_clean.head())

Clean Video Game Dataset Shape: (16327, 11)


,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [22]:
superstore_yearly = (
    sales.groupby("Year", as_index=False)["Sales"]
    .sum()
    .rename(columns={"Sales": "Superstore_Sales"})
)

gaming_yearly = (
    vgsales_clean.groupby("Year", as_index=False)["Global_Sales"]
    .sum()
)

multi_source = pd.merge(
    superstore_yearly,
    gaming_yearly,
    on="Year",
    how="left"
)

display(multi_source)

,Year,Superstore_Sales,Global_Sales
0,2015,479856.2081,264.44
1,2016,459436.0054,70.93
2,2017,600192.5500,0.05
3,2018,722052.0192,NaN


In [23]:
multi_source["Superstore_Sales_Index"] = (
    multi_source["Superstore_Sales"]
    / multi_source["Superstore_Sales"].max()
    * 100
)

multi_source["Gaming_Sales_Index"] = (
    multi_source["Global_Sales"]
    / multi_source["Global_Sales"].max()
    * 100
)

display(multi_source)

,Year,Superstore_Sales,Global_Sales,Superstore_Sales_Index,Gaming_Sales_Index
0,2015,479856.2081,264.44,66.457291,100.000000
1,2016,459436.0054,70.93,63.629211,26.822720
2,2017,600192.5500,0.05,83.123173,0.018908
3,2018,722052.0192,NaN,100.000000,NaN


In [ ]:
#Weekly and monthly aggregations can provide insights into sales trends and seasonality. Let's compute the daily sales from the primary sales dataset.
daily_sales = (
    sales
    .set_index("Order Date")
    .resample("D")["Sales"]
    .sum()
    .reset_index()
)

display(daily_sales.head())

,Order Date,Sales
0,2015-01-03,16.448
1,2015-01-04,288.060
2,2015-01-05,19.536
3,2015-01-06,4407.100
4,2015-01-07,87.158
